In [0]:
%pip install xgboost scikit-learn

In [0]:
import mlflow
import xgboost as xgb
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

# Use Databricks Unity Catalog as the MLflow model registry
mlflow.set_registry_uri("databricks-uc")

In [0]:
df = (
    spark.table("stocks.gold.stocks_combined_features")
    .dropna()          # drop rows where FX/macro not yet available
    .orderBy("Date")   # ensure chronological order before split
    .toPandas()
)

EXCLUDE = ["label", "Date", "company"]
feature_cols = [c for c in df.columns if c not in EXCLUDE]

X = df[feature_cols]
y = df["label"]

# Chronological 80/20 split — never shuffle time series
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Training rows: {len(X_train)}  |  Test rows: {len(X_test)}")
print(f"Features: {len(feature_cols)}")

In [0]:
mlflow.xgboost.autolog()

with mlflow.start_run(run_name="combined_model_v1") as run:
    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        use_label_encoder=False,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False,
    )

    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    mlflow.log_metrics({"accuracy": acc, "roc_auc": auc})

    # Log the exact feature list — predict notebook reads this back to guarantee column order
    mlflow.log_param("feature_cols", str(feature_cols))

    # Register in Unity Catalog model registry
    mlflow.xgboost.log_model(
        model,
        artifact_path="model",
        registered_model_name="stocks.models.combined_predictor",
    )

    print(f"Run ID : {run.info.run_id}")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

In [0]:
# Top-20 feature importances — verify FX and macro features appear
import pandas as pd
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.nlargest(20).sort_values().plot.barh(figsize=(8, 8), title="Top 20 Feature Importances")
display()

In [0]:
# After running this notebook, go to Databricks > Models > stocks.models.combined_predictor
# and set the 'champion' alias on the newly registered version before running predict_combined.